In [ ]:
import os
import sys
import pathlib
import importlib
import textwrap

from IPython import get_ipython
shell = get_ipython()

import importlib.metadata
from packaging.requirements import Requirement

def check_requirements(file_path):
    with open(file_path) as f:
        # Read file, ignore empty lines and comments
        lines = [line.strip() for line in f if line.strip() and not line.startswith('#')]

    no_conflict = True
    for line in lines:
        req = Requirement(line)
        try:
            # Check if the package exists
            installed_version = importlib.metadata.version(req.name)

            # Check if the installed version matches the requirements.txt specifiers
            if not req.specifier.contains(installed_version):
                print(f"⚠️ Conflict: {req.name} requires {req.specifier}, but {installed_version} is installed.")
                no_conflict = False

        except importlib.metadata.PackageNotFoundError:
            print(f"❌ Missing: {req.name}")
            no_conflict = False

    if no_conflict:
        print("✅ All packages and versions are perfectly matched!")

def bootstrap_environment():
    global shell
    # Initialize default secrets
    api_key, hf_token = None, None

    # Detect active Python kernel version to prevent C++ ABI mismatches
    py_version = f"{sys.version_info.major}.{sys.version_info.minor}"
    cad_site_packages = f"/opt/cad_env/lib/python{py_version}/site-packages"

    # Detect Colab environment
    IN_COLAB = 'COLAB_GPU' in os.environ
    if IN_COLAB:
        print("Running in Google Colab")
        from google.colab import drive, userdata
        drive.mount('/content/drive')

        # Persistent library setup
        root = pathlib.Path("/content/drive/MyDrive/Senior-Capstone")

        api_key = userdata.get("GEMINI_API_KEY")
        hf_token = userdata.get("HF_TOKEN")

        # Install pythonocc-core via Micromamba (No Kernel Restart Needed)
        cad_env_archive = root / f"cad_env_backup_py{py_version}.tar.xz"

        if not os.path.exists("/opt/cad_env"):
            if cad_env_archive.exists():
                print("📦 Restoring CAD environment from Google Drive cache...")
                # Unzip the archive directly into the root directory (re-creating /opt/cad_env)
                shell.system(f"tar -xJf {cad_env_archive} -C /")
                shell.system('echo "/opt/cad_env/lib" > /etc/ld.so.conf.d/cad_env.conf')
                shell.system("ldconfig")
            else:
                print("⬇️ Installing pythonocc-core via Micromamba (First Time Download)...")
                shell.system("wget -qO- https://micro.mamba.pm/api/micromamba/linux-64/latest | tar -xvj bin/micromamba")
                shell.system(f"MAMBA_ROOT_PREFIX=/tmp/mamba_root ./bin/micromamba create -y -p /opt/cad_env -c conda-forge python={py_version} pythonocc-core")
                shell.system('echo "/opt/cad_env/lib" > /etc/ld.so.conf.d/cad_env.conf')
                shell.system("ldconfig")
                shell.system(textwrap.dedent("""/opt/cad_env/bin/pip install \
                        "ipython==7.34.0" \
                        "google-auth==2.47.0" \
                        "ipykernel==6.17.1" \
                        "pandas==2.2.2" \
                        "requests==2.32.4" \
                        "tornado==6.5.1" \
                        "cryptography<44" \
                        "numpy<2.1" \
                        "fsspec==2025.3.0" \
                        "websockets<16.0" \
                        "pillow<12.0" \
                        "docutils<0.22" """))
                print(f"💾 Backing up environment to Google Drive: {cad_env_archive.name}")
                # Compress the local SSD folder into a single file on Google Drive
                shell.system(f"tar -cJf {cad_env_archive} -C / opt/cad_env")
            shell.kernel.do_shutdown(True)

        # Link the isolated environment
        if cad_site_packages not in sys.path:
            sys.path.append(cad_site_packages)
            print(f"Linked isolated CAD environment ({py_version})")

    # Detect Kaggle environment
    elif 'KAGGLE_KERNEL_RUN_TYPE' in os.environ:
        print("Running in Kaggle")
        from kaggle_secrets import UserSecretsClient
        user_secrets = UserSecretsClient()
        api_key = user_secrets.get_secret("GEMINI_API_KEY")
        hf_token = user_secrets.get_secret("HF_TOKEN")

        root = pathlib.Path("/kaggle/working/")

        # Kaggle specific installs
        shell.system("wget https://raw.githubusercontent.com/hanssbtn/Senior-Capstone/main/requirements.txt")

        # Install pythonocc-core via Micromamba (Bypassing native Conda to ensure strict isolation)
        if not os.path.exists("/opt/cad_env"):
            print("Installing pythonocc-core via Micromamba...")
            shell.system("wget -qO- https://micro.mamba.pm/api/micromamba/linux-64/latest | tar -xvj bin/micromamba")
            shell.system(f"MAMBA_ROOT_PREFIX=/tmp/mamba_root ./bin/micromamba create -y -p /opt/cad_env -c conda-forge python={py_version} pythonocc-core")
            shell.system('echo "/opt/cad_env/lib" > /etc/ld.so.conf.d/cad_env.conf')
            shell.system("ldconfig")
            shell.kernel.do_shutdown(True)

        # Link the isolated environment
        if cad_site_packages not in sys.path:
            sys.path.append(cad_site_packages)
            print(f"Linked isolated CAD environment ({py_version})")

    # Detect local environment
    else:
        print("Running locally")
        root = pathlib.Path(".")
        try:
            from dotenv import load_dotenv
            load_dotenv(str(root / "secret.env")) # Insert Huggingface token here
            api_key = os.getenv("GEMINI_API_KEY")
            hf_token = os.getenv("HF_TOKEN")
        except ImportError:
            print("Tip: Install 'python-dotenv' for local secret management")

    # Core Requirements processing
    req_path = root / "requirements-cloud.txt" if IN_COLAB else "requirements.txt"
    if req_path.exists() and not check_requirements(req_path):
        shell.system(f"pip install -r {str(req_path)} -q --ignore-installed")
        pass

    print("Finished bootstrapping environment.")

    dataset_dir = root / "dataset"
    model_dir = root / "models"

    os.makedirs(dataset_dir, exist_ok=True)
    os.makedirs(model_dir, exist_ok=True)

    return root, dataset_dir, model_dir, api_key, hf_token

ROOT_DIR, DATASET_DIR, MODEL_DIR, API_KEY, HF_TOKEN = bootstrap_environment()
del shell

Running in Google Colab
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
⬇️ Installing pythonocc-core via Micromamba (First Time Download)...
bin/micromamba
[+] 0.0s
[+] 0.1s
Fetch Shard Index for conda-forge/linux-64                                                ✔ Done (0.1 sec)
Fetch Shard Index for conda-forge/noarch                                                  ✔ Done (0.1 sec)
Fetching and Parsing Packages' Shards                                                    ✔ Done (11.5 sec)
Using Cached Shard Index for conda-forge/linux-64                                                   ✔ Done
Using Cached Shard Index for conda-forge/noarch                                                     ✔ Done
Fetching and Parsing Packages' Shards                                                     ✔ Done (1.0 sec)

Resolving Environment                                                                    ✔ Done (12.8 

In [ ]:
import pathlib
import sys
ROOT_DIR = pathlib.Path("/content/drive/MyDrive/Senior-Capstone")
COLAB_LIBS = ROOT_DIR / "colab_libs"
if str(COLAB_LIBS) not in sys.path:
    sys.path.append(str(COLAB_LIBS))

!pip list | grep -E "ipython|google-auth|ipykernel|pandas|numpy"

In [14]:
import importlib.metadata
from packaging.requirements import Requirement
from packaging.utils import canonicalize_name

def check_environment_compatibility():
    # 1. Map out everything currently installed in the environment
    # canonicalize_name ensures 'Google-Auth' and 'google_auth' are treated the same
    installed_packages = {
        canonicalize_name(dist.metadata['Name']): dist.version
        for dist in importlib.metadata.distributions()
        if dist.metadata['Name'] is not None
    }

    conflicts = []
    missing = []

    # 2. Loop through every installed package
    for dist in importlib.metadata.distributions():
        package_name = dist.metadata['Name']
        requirements = dist.requires

        if not requirements:
            continue

        # 3. Check each requirement for that package
        for req_str in requirements:
            try:
                req = Requirement(req_str)

                # Skip requirements that don't apply to the OS/Environment (e.g., Windows-only packages)
                if req.marker and not req.marker.evaluate():
                    continue

                req_name = canonicalize_name(req.name)

                # Check if the required package is installed
                if req_name not in installed_packages:
                    missing.append(f"❌ {package_name} requires '{req.name}', which is not installed.")
                    continue

                # Check if the installed version matches the requirement
                installed_version = installed_packages[req_name]
                if not req.specifier.contains(installed_version, prereleases=True):
                    conflicts.append(
                        f"⚠️ CONFLICT: {package_name} requires '{req}', "
                        f"but you have version {installed_version} installed."
                    )
            except Exception:
                # Catch poorly formatted requirement strings in older packages
                pass

    # 4. Print the results
    if not conflicts and not missing:
        print("✅ No conflicts found.")
    else:
        print("\n--- Missing Dependencies ---")
        for m in missing: print(m)

        print("\n--- Version Conflicts ---")
        for c in conflicts: print(c)

check_environment_compatibility()


--- Missing Dependencies ---
❌ qwen-vl-utils requires 'av', which is not installed.
❌ trame requires 'trame-server', which is not installed.
❌ trame requires 'trame-client', which is not installed.
❌ trame requires 'trame-common', which is not installed.
❌ ipython requires 'ipython-pygments-lexers', which is not installed.
❌ stack-data requires 'executing', which is not installed.
❌ stack-data requires 'asttokens', which is not installed.
❌ stack-data requires 'pure-eval', which is not installed.
❌ trame-components requires 'trame-client', which is not installed.
❌ cyclopts requires 'rich-rst', which is not installed.
❌ nvidia-cusparse requires 'nvidia-nvjitlink', which is not installed.
❌ torch requires 'cuda-toolkit', which is not installed.
❌ torch requires 'cuda-bindings', which is not installed.
❌ torch requires 'nvidia-nvshmem-cu13', which is not installed.
❌ torch requires 'triton', which is not installed.

--- Version Conflicts ---
⚠️ CONFLICT: google-colab requires 'google-au

In [17]:
# --- OpenCascade Imports ---
from OCC.Extend.DataExchange import read_step_file
from OCC.Core.BRepMesh import BRepMesh_IncrementalMesh
from OCC.Core.StlAPI import StlAPI_Writer
from OCC.Core.Bnd import Bnd_Box
from OCC.Core.BRepBndLib import brepbndlib_Add

from OCC.Core.STEPControl import STEPControl_Reader
from OCC.Core.TopExp import TopExp_Explorer
from OCC.Core.TopAbs import TopAbs_FACE, TopAbs_EDGE, TopAbs_VERTEX
from OCC.Core.TopoDS import topods
from OCC.Core.BRepAdaptor import BRepAdaptor_Surface, BRepAdaptor_Curve
from OCC.Core.BRep import BRep_Tool
from OCC.Core.GeomAbs import (
    GeomAbs_Plane, GeomAbs_Cylinder, GeomAbs_Cone, GeomAbs_Sphere,
    GeomAbs_Torus, GeomAbs_BSplineSurface, GeomAbs_SurfaceOfExtrusion,
    GeomAbs_SurfaceOfRevolution, GeomAbs_Line, GeomAbs_Circle,
    GeomAbs_Ellipse, GeomAbs_Parabola, GeomAbs_Hyperbola, GeomAbs_BSplineCurve
)
from OCC.Core.BRepGProp import brepgprop
from OCC.Core.GProp import GProp_GProps
from OCC.Core.BRepBuilderAPI import BRepBuilderAPI_Transform
from OCC.Core.gp import gp_Trsf, gp_Vec

import json
import gc
import os
import torch
import trimesh
import io
from pathlib import Path
from PIL import Image
from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor
from qwen_vl_utils import process_vision_info

import torch.nn as nn
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model
import re
import glob

import numpy as np

import networkx as nx
from networkx.algorithms import isomorphism

import math

import torch.nn as nn
from torch.amp import autocast, GradScaler
from torch.utils.data import Dataset, DataLoader, random_split
from peft import LoraConfig, get_peft_model, TaskType, PeftModel
import matplotlib.pyplot as plt
from tqdm import tqdm
import evaluate
import math
import traceback


ModuleNotFoundError: No module named 'qwen_vl_utils'

In [ ]:

import torch.nn as nn
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model
import re
import glob

# Log in to Hugging Face
from huggingface_hub import snapshot_download, login
login(token=HF_TOKEN)

del HF_TOKEN

def download_model(model_name, model_id):
    local_dir = MODEL_DIR / model_name
    os.makedirs(local_dir, exist_ok=True)
    with os.scandir(local_dir) as it:
        has_single_file = (local_dir / "model.safetensors").exists()
        has_sharded_files = (local_dir / "model.safetensors.index.json").exists() and any(glob.iglob("*.safetensors", root_dir=local_dir))

        if not (has_single_file or has_sharded_files):
            print(f"Downloading {model_id}...")
            download_path = snapshot_download(
                repo_id=model_id,
                local_dir=local_dir,
                ignore_patterns=["*.msgpack", "*.h5", "*.ot", "rust_model.ot", "original/*", "*.flax", "README.md"],
                max_workers=4
            )
            print(f"Download complete! Saved to: {download_path}")
        else:
            print(f"Model '{model_id}' is already fully downloaded at {local_dir}.")

download_model("qwen3.5-2B", "Qwen/Qwen3.5-2B")
download_model("roberta-base", "roberta-base")
download_model("qwen2.5-VL-7B", "Qwen/Qwen2.5-VL-7B-Instruct")

Model 'Qwen/Qwen2.5-VL-7B-Instruct' is already fully downloaded at /content/drive/MyDrive/Senior-Capstone/models/qwen2.5-VL-7B.


# Dataset Preprocessing

## 1. Automatic Annotation Generation using Qwen2.5-VL-7B

In [ ]:
import json
import gc
import os
import torch
import trimesh
import io
from pathlib import Path
from PIL import Image
from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor
from qwen_vl_utils import process_vision_info

# --- OpenCascade Imports ---
from OCC.Extend.DataExchange import read_step_file
from OCC.Core.BRepMesh import BRepMesh_IncrementalMesh
from OCC.Core.StlAPI import StlAPI_Writer
from OCC.Core.Bnd import Bnd_Box
from OCC.Core.BRepBndLib import brepbndlib_Add

# --- Configuration ---
DATASET_DIR = ROOT_DIR / "dataset" / "abc_0000_step_v00"
OUTPUT_FILE = ROOT_DIR / "annotated_dataset_vlm.jsonl"
RENDER_DIR = ROOT_DIR / "dataset_renders" # Saves images permanently for later verification

os.makedirs(RENDER_DIR, exist_ok=True)

# =====================================================================
# PHASE 1: CALCULATE BOUNDING BOX
# =====================================================================
def get_actual_dimensions(shape):
    """Calculates the true bounding box dimensions using the OpenCascade kernel."""
    bbox = Bnd_Box()
    bbox.SetGap(1.e-5)
    brepbndlib_Add(shape, bbox)

    xmin, ymin, zmin, xmax, ymax, zmax = bbox.Get()
    dx = round(xmax - xmin, 2)
    dy = round(ymax - ymin, 2)
    dz = round(zmax - zmin, 2)

    dims = sorted([dx, dy, dz], reverse=True)
    return f"{dims[0]} mm x {dims[1]} mm x {dims[2]} mm"

# =====================================================================
# PHASE 2: RENDER EACH IMAGE (Saves to Drive)
# =====================================================================
def generate_notebook_render(step_shape, output_path):
    """Renders an isometric view natively in Jupyter using Trimesh."""
    try:
        BRepMesh_IncrementalMesh(step_shape, 0.1).Perform()
        writer = StlAPI_Writer()
        temp_stl = "temp_render.stl"
        writer.Write(step_shape, temp_stl)

        mesh = trimesh.load(temp_stl)
        scene = trimesh.Scene(mesh)

        png_data = scene.save_image(resolution=[512, 512], visible=True)
        image = Image.open(io.BytesIO(png_data))

        bg = Image.new("RGB", image.size, (255, 255, 255))
        if len(image.split()) == 4:
            bg.paste(image, mask=image.split()[3])
        else:
            bg.paste(image)

        bg.save(output_path)
        os.remove(temp_stl)
        return True
    except Exception as e:
        print(f"  -> [!] Render failed: {e}")
        if os.path.exists("temp_render.stl"):
            os.remove("temp_render.stl")
        return False

# =====================================================================
# PHASE 3: QWEN ANNOTATION
# =====================================================================
gc.collect()
torch.cuda.empty_cache()

print("Loading Qwen2.5-VL-7B")
VLM_MODEL_DIR = str(MODEL_DIR / "qwen2.5-VL-7B")
vlm_processor = AutoProcessor.from_pretrained(VLM_MODEL_DIR)
vlm_model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    VLM_MODEL_DIR,
    device_map="auto",
    torch_dtype=torch.float16,
    load_in_4bit=True
)

def get_qwen_annotation(image_path, true_bbox):
    prompt_text = f"""You are an expert mechanical engineer. Read ALL attached raw CAD STEP files.

Analyze the Boundary Representation (B-Rep) geometry for EACH file and extract the hierarchical annotations.

FACTUAL BOUNDING BOX: {true_bbox}

Output STRICTLY a JSON object with no markdown wrappers:
{{
  "L1": "Probabilistic functional category; the specific mechanical function or classification of the part (e.g., 'Hexagonal Bolt', 'Motor Mounting Bracket')",
  "L2": "The macroscopic B-Rep topology and geometric features with information from [L1] annotation for added context (e.g., 'A cylindrical shaft joined to a hexagonal head').",
  "L3": "A complete description of the object represented by the STEP file, containing parametric constraints & dimensions, along with information gathered from the previous levels [L1] and [L2], such as the what from [L1] and the brief description from [L2]. You MUST include the exact factual bounding box provided above."
}}

Example Expected Output:
{{
    "L1": "Dowel Alignment Pin",
    "L2": "A slender cylindrical solid with slightly radiused edges on its terminal planar faces.",
    "L3": "Precision locating pin with a 32.5 mm length and 15.0 mm nominal diameter. Bounding box: 32.5 mm x 15.0 mm x 15.0 mm."
}}
{{
    "L1": "Annular Shim",
    "L2": "A very thin, flat circular disk with a concentric internal cutout forming a simple continuous ring.",
    "L3": "Precision spacing shim with an outer diameter of 25.0 mm and a thickness of 0.6 mm. Bounding box: 25.0 mm x 25.0 mm x 0.6 mm."
}}"""

    messages = [
        {"role": "system", "content": "You are a mechanical engineering AI. Output raw JSON only."},
        {"role": "user", "content": [
            {"type": "image", "image": image_path},
            {"type": "text", "text": prompt_text}
        ]}
    ]

    text = vlm_processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    image_inputs, video_inputs = process_vision_info(messages)
    inputs = vlm_processor(
        text=[text],
        images=image_inputs,
        videos=video_inputs,
        padding=True,
        return_tensors="pt"
    ).to("cuda")

    with torch.no_grad():
        generated_ids = vlm_model.generate(**inputs, max_new_tokens=2048)

    generated_ids_trimmed = [out_ids[len(in_ids):] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)]
    output_text = vlm_processor.batch_decode(generated_ids_trimmed, skip_special_tokens=True)[0]

    try:
        clean_json = output_text.replace("```json", "").replace("```", "").strip()
        return json.loads(clean_json)
    except Exception as e:
        print(f"  -> [!] Failed to parse Qwen JSON: {output_text}")
        return None

# =====================================================================
# PHASE 4: BOUNDING BOX VERIFICATION
# =====================================================================
def verify_annotation(annotation_dict, true_bbox):
    """Checks if Qwen adhered to the strict Math Anchor constraint."""
    l3_text = annotation_dict.get("L3", "")
    if not l3_text:
        return False, "No L3 Annotation."
    if true_bbox not in l3_text:
        return False, f"Hallucinated dimensions. Expected {true_bbox}"
    return True, "Math anchor verified."

# =====================================================================
# MAIN EXECUTION LOOP WITH CONTINUATION LOGIC
# =====================================================================
step_files = list(DATASET_DIR.rglob("*.step"))
print(f"Found {len(step_files)} STEP files in total.")

# --- Continuation Logic ---
processed_files = set()
if OUTPUT_FILE.exists():
    with open(OUTPUT_FILE, 'r') as f:
        for line in f:
            if line.strip():
                try:
                    data = json.loads(line)
                    orig_filename = Path(data["file_path"]).name
                    processed_files.add(orig_filename)
                except json.JSONDecodeError:
                    continue

print(f"Resuming progress: {len(processed_files)} files already processed.")

LIMIT = 2040
processed_count = 0

with open(OUTPUT_FILE, 'a') as f_out:
    for file_path in step_files:
        if processed_count >= LIMIT:
            print(f"Reached limit of {LIMIT} processed files.")
            break

        filename = file_path.name

        # Skip if already processed in a previous session
        if filename in processed_files:
            continue

        # Define persistent image path
        image_path = RENDER_DIR / f"{file_path.stem}.png"

        try:
            file_size_mb = os.path.getsize(file_path) / (1024 * 1024)
            if file_size_mb > 2.5:
                print(f"  -> [!] Skipping {filename}: File too large ({file_size_mb:.2f} MB)")
                continue

            processed_count += 1
            print(f"\n[{processed_count}/{LIMIT}] Processing {file_path.parent.name}/{filename}")

            # 1. Math Anchor
            shape = read_step_file(str(file_path))
            true_bbox = get_actual_dimensions(shape)

            # 2. Render Eyes (Skip if image already exists from an aborted run)
            if not image_path.exists():
                if not generate_notebook_render(shape, str(image_path)):
                    continue
            else:
                print(f"  -> Using existing render: {image_path.name}")

            # 3. Generate Annotation
            annotation_dict = get_qwen_annotation(str(image_path), true_bbox)
            if not annotation_dict:
                continue

            l1_target = annotation_dict.get("L1", "Unknown")
            print(f"  -> Qwen predicts: {l1_target}")

            # 4. Local Verification
            is_valid, reason = verify_annotation(annotation_dict, true_bbox)
            if not is_valid:
                print(f"  -> [!] MISMATCH FLAGGED: {reason}")
            else:
                print(f"  -> Verification: PASSED")

            # 5. Save Results
            dataset_entry = {
                "file_path": f"dataset/abc_0000_step_v00/{filename}",
                "annotation": f"[L1] {annotation_dict['L1']} [L2] {annotation_dict['L2']} [L3] {annotation_dict['L3']}",
                "verification_flag": not is_valid
            }

            f_out.write(json.dumps(dataset_entry, separators=(',', ':')) + "\n")
            f_out.flush()

        except Exception as e:
            print(f"Error processing {filename}: {e}")

## 2. Canonical Graph Normalization

In [ ]:
import numpy as np
from OCC.Core.STEPControl import STEPControl_Reader
from OCC.Core.TopExp import TopExp_Explorer
from OCC.Core.TopAbs import TopAbs_FACE, TopAbs_EDGE, TopAbs_VERTEX
from OCC.Core.TopoDS import topods
from OCC.Core.BRepAdaptor import BRepAdaptor_Surface, BRepAdaptor_Curve
from OCC.Core.BRep import BRep_Tool
from OCC.Core.GeomAbs import (
    GeomAbs_Plane, GeomAbs_Cylinder, GeomAbs_Cone, GeomAbs_Sphere,
    GeomAbs_Torus, GeomAbs_BSplineSurface, GeomAbs_SurfaceOfExtrusion,
    GeomAbs_SurfaceOfRevolution, GeomAbs_Line, GeomAbs_Circle,
    GeomAbs_Ellipse, GeomAbs_Parabola, GeomAbs_Hyperbola, GeomAbs_BSplineCurve
)
from OCC.Core.BRepGProp import brepgprop
from OCC.Core.GProp import GProp_GProps
from OCC.Core.BRepBuilderAPI import BRepBuilderAPI_Transform
from OCC.Core.gp import gp_Trsf, gp_Vec

import networkx as nx
from networkx.algorithms import isomorphism

import math

def canonical_normalize_shape(shape):
    """
    Translates the shape's centroid to (0,0,0) and rotates it
    using PCA so the principal axis of variance aligns with the Z-axis.
    """
    # 1. Extract all vertices to build a mathematical Point Cloud
    vertices = []
    explorer = TopExp_Explorer(shape, TopAbs_VERTEX)
    while explorer.More():
        v = topods.Vertex(explorer.Current())
        pnt = BRep_Tool.Pnt(v)
        vertices.append([pnt.X(), pnt.Y(), pnt.Z()])
        explorer.Next()

    # Fallback for extremely simple shapes (e.g., a perfect sphere has 1 vertex in B-Rep)
    if len(vertices) < 3:
        return shape

    P = np.array(vertices)

    # 2. Centroid Subtraction (Translational Invariance)
    mu = np.mean(P, axis=0)
    P_centered = P - mu

    # 3. PCA Covariance & Eigen Decomposition (Rotational Invariance)
    covariance_matrix = np.cov(P_centered, rowvar=False)
    _, eigenvectors = np.linalg.eigh(covariance_matrix)

    # np.linalg.eigh returns eigenvalues in ASCENDING order.
    # By keeping [0, 1, 2], the longest axis (index 2) automatically aligns to the Z-axis.

    # Ensure a right-handed coordinate system (Determinant must be strictly positive 1)
    if np.linalg.det(eigenvectors) < 0:
        eigenvectors[:, 0] = -eigenvectors[:, 0]

    # The eigenvectors represent the rotation FROM the canonical frame TO the current frame.
    # Use the inverse (transpose) to rotate the object TO the canonical frame.
    R = eigenvectors.T

    # 4. Build the OpenCASCADE Transformation Matrices
    # Translation: Move the object by -mu
    trsf_translate = gp_Trsf()
    trsf_translate.SetTranslation(gp_Vec(-mu[0], -mu[1], -mu[2]))

    # Rotation: Apply the transposed eigenvector matrix
    trsf_rotate = gp_Trsf()
    trsf_rotate.SetValues(
        R[0,0], R[0,1], R[0,2], 0.0,
        R[1,0], R[1,1], R[1,2], 0.0,
        R[2,0], R[2,1], R[2,2], 0.0
    )

    # Combine transformations: Translate first, then Rotate
    final_trsf = trsf_rotate.Multiplied(trsf_translate)

    # 5. Get the transformed shape
    transformer = BRepBuilderAPI_Transform(shape, final_trsf, True)
    normalized_shape = transformer.Shape()

    return normalized_shape

## 3. Topological Linearization

In [ ]:
def get_surface_token(geom_type):
    """Maps OpenCASCADE surfaces to exact ISO 10303 tokens."""
    mapping = {
        GeomAbs_Plane: "<PLANE>",
        GeomAbs_Cylinder: "<CYLINDRICAL_SURFACE>",
        GeomAbs_Cone: "<CONICAL_SURFACE>",
        GeomAbs_Sphere: "<SPHERICAL_SURFACE>",
        GeomAbs_Torus: "<TOROIDAL_SURFACE>",
        GeomAbs_BSplineSurface: "<B_SPLINE_SURFACE_WITH_KNOTS>",
        GeomAbs_SurfaceOfExtrusion: "<SURFACE_OF_LINEAR_EXTRUSION>",
        GeomAbs_SurfaceOfRevolution: "<SURFACE_OF_REVOLUTION>"
    }
    return mapping.get(geom_type, "<ADVANCED_FACE>")

def get_curve_token(geom_type):
    """Maps OpenCASCADE curves to exact ISO 10303 tokens."""
    mapping = {
        GeomAbs_Line: "<LINE>",
        GeomAbs_Circle: "<CIRCLE>",
        GeomAbs_Ellipse: "<ELLIPSE>",
        GeomAbs_Parabola: "<PARABOLA>",
        GeomAbs_Hyperbola: "<HYPERBOLA>",
        GeomAbs_BSplineCurve: "<B_SPLINE_CURVE_WITH_KNOTS>"
    }
    return mapping.get(geom_type, "<EDGE_CURVE>")

def calculate_geometric_invariants(occ_shape):
    """Calculates Area/Length and Centroid (X,Y,Z) for the sorting key."""
    props = GProp_GProps()
    shape_type = occ_shape.ShapeType()

    magnitude: float = 0.0
    centroid: tuple[float, float, float] = (.0, .0, .0)

    if shape_type == TopAbs_FACE:
        brepgprop.SurfaceProperties(occ_shape, props)
        magnitude = props.Mass() # Return surface area for faces
        com = props.CentreOfMass()
        centroid = (com.X(), com.Y(), com.Z())
    elif shape_type == TopAbs_EDGE:
        brepgprop.LinearProperties(occ_shape, props)
        magnitude = props.Mass() # Return curve length for edges
        com = props.CentreOfMass()
        centroid = (com.X(), com.Y(), com.Z())
    elif shape_type == TopAbs_VERTEX:
        # Vertices have 0 magnitude
        from OCC.Core.BRep import BRep_Tool
        pnt = BRep_Tool.Pnt(occ_shape)
        centroid = (pnt.X(), pnt.Y(), pnt.Z())


    return magnitude, centroid

def get_sorting_key(graph, node):
    data = graph.nodes[node]

    # Use pre-computed raw math to sort the graph
    math_vec = data.get('macro_math', data.get('raw_math', np.zeros(128, dtype=np.float32)))
    centroid = math_vec[0:3]
    magnitude = math_vec[6]

    dist_to_origin = math.sqrt(sum(c**2 for c in centroid))
    degree = graph.degree(node)

    return (-magnitude, dist_to_origin, -degree, -centroid[2], -centroid[1], -centroid[0])

## Extract Geometric Vectors (Stream B)

In [ ]:
def extract_stream_b_vector(occ_shape):
    """Calculates the 128-dim Stream B vector for raw primitives."""
    v = np.zeros(128, dtype=np.float32)
    if occ_shape is None:
        return v

    magnitude, centroid = calculate_geometric_invariants(occ_shape)
    v[0:3] = centroid
    v[6] = magnitude

    try:
        if occ_shape.ShapeType() == TopAbs_FACE:
            surf = BRepAdaptor_Surface(topods.Face(occ_shape), True)
            if surf.GetType() == GeomAbs_Cylinder:
                v[7] = surf.Cylinder().Radius()
                axis = surf.Cylinder().Axis().Direction()
                v[3:6] = [axis.X(), axis.Y(), axis.Z()]
            elif surf.GetType() == GeomAbs_Cone:
                v[7] = surf.Cone().RefRadius()
                v[8] = surf.Cone().SemiAngle()
        elif occ_shape.ShapeType() == TopAbs_EDGE:
            curve = BRepAdaptor_Curve(topods.Edge(occ_shape))
            if curve.GetType() == GeomAbs_Circle:
                v[7] = curve.Circle().Radius()
    except: pass
    return v

def precompute_geometric_vectors(brep_graph):
    for node, data in brep_graph.nodes(data=True):
        if data.get('raw_math', None):
            print(f"Skipping {node}")
            continue
        occ_shape = data.get('occ_shape')
        # entity_type = data.get('entity_type')

        vector = extract_stream_b_vector(occ_shape)

        # Store it directly in the NetworkX node attributes
        brep_graph.nodes[node]['raw_math'] = vector

    return brep_graph

## Semantic Macro Compression

In [ ]:
def node_match(n1, n2):
    return n1.get('entity_type') == n2.get('entity_type')

def compress_hole_macros(brep_graph):
    hole_template = nx.DiGraph()
    hole_template.add_node("cyl", entity_type="<CYLINDRICAL_SURFACE>")
    hole_template.add_node("c1", entity_type="<CIRCLE>")
    hole_template.add_node("c2", entity_type="<CIRCLE>")
    hole_template.add_node("p1", entity_type="<PLANE>")
    hole_template.add_node("p2", entity_type="<PLANE>")

    # The cylinder and planes share the circular edges
    hole_template.add_edges_from([("cyl", "c1"), ("cyl", "c2"), ("p1", "c1"), ("p2", "c2")])

    matcher = isomorphism.DiGraphMatcher(brep_graph, hole_template, node_match=node_match)

    macro_id_counter = 1
    for match in list(matcher.subgraph_isomorphisms_iter()):
        nodes_in_match = list(match.keys())
        # Check if another macro already consumed these nodes
        if not all(n in brep_graph for n in nodes_in_match):
            continue

        inv_match = {v: k for k, v in match.items()}
        cyl_math = np.copy(brep_graph.nodes[inv_match["cyl"]]['raw_math'])
        p1_math = brep_graph.nodes[inv_match["p1"]]['raw_math']
        p2_math = brep_graph.nodes[inv_match["p2"]]['raw_math']

        # Euclidean distance between the two plane centroids = Hole Depth
        depth = np.linalg.norm(p1_math[0:3] - p2_math[0:3])
        cyl_math[9] = depth # Store depth in slot 9

        # Assign occ_shape=None so the get_sorting_key function doesn't crash
        # when trying to calculate the surface area of a macro node.
        new_macro_id = f"MACRO_HOLE_{macro_id_counter}"
        brep_graph.add_node(new_macro_id, entity_type="<THROUGH_HOLE>", occ_shape=None, macro_math=cyl_math)
        macro_id_counter += 1

        # Recalculate surrounding topology to the new macro node
        for old_node in nodes_in_match:
            for predecessor in list(brep_graph.predecessors(old_node)):
                if predecessor not in nodes_in_match:
                    brep_graph.add_edge(predecessor, new_macro_id)
            for successor in list(brep_graph.successors(old_node)):
                if successor not in nodes_in_match:
                    brep_graph.add_edge(new_macro_id, successor)

        brep_graph.remove_nodes_from(nodes_in_match)

    if macro_id_counter > 1:
        print(f"  -> Compressed {macro_id_counter - 1} <THROUGH_HOLE> macros.")
    return brep_graph

def compress_chamfer_macros(brep_graph):
    chamfer_template = nx.DiGraph()
    chamfer_template.add_node("cone", entity_type="<CONICAL_SURFACE>")
    chamfer_template.add_node("c1", entity_type="<CIRCLE>")
    chamfer_template.add_node("c2", entity_type="<CIRCLE>")
    chamfer_template.add_node("p1", entity_type="<PLANE>")
    chamfer_template.add_node("p2", entity_type="<PLANE>")

    # Topology: Cone and Planes share the circular edges
    chamfer_template.add_edges_from([("cone", "c1"), ("cone", "c2"), ("p1", "c1"), ("p2", "c2")])

    matcher = isomorphism.DiGraphMatcher(brep_graph, chamfer_template, node_match=node_match)

    macro_id_counter = 1
    for match in list(matcher.subgraph_isomorphisms_iter()):
        nodes_in_match = list(match.keys())
        # Check if another macro already consumed these nodes
        if not all(n in brep_graph for n in nodes_in_match):
            continue

        inv_match = {v: k for k, v in match.items()}
        chamfer_math = np.copy(brep_graph.nodes[inv_match["cone"]]['raw_math'])

        # Calculate the chamfer width (distance between plane centroids)
        p1_math = brep_graph.nodes[inv_match["p1"]]['raw_math']
        p2_math = brep_graph.nodes[inv_match["p2"]]['raw_math']
        chamfer_math[9] = np.linalg.norm(p1_math[0:3] - p2_math[0:3])

        # Assign occ_shape=None so the get_sorting_key function doesn't crash
        # when trying to calculate the surface area of a macro node.
        new_macro_id = f"MACRO_CHAMFER_{macro_id_counter}"
        brep_graph.add_node(new_macro_id, entity_type="<CHAMFER_EDGE>", occ_shape=None, macro_math=chamfer_math)
        macro_id_counter += 1

        # Recalculate surrounding topology to the new macro node
        for old_node in nodes_in_match:
            for predecessor in list(brep_graph.predecessors(old_node)):
                if predecessor not in nodes_in_match:
                    brep_graph.add_edge(predecessor, new_macro_id)
            for successor in list(brep_graph.successors(old_node)):
                if successor not in nodes_in_match:
                    brep_graph.add_edge(new_macro_id, successor)

        brep_graph.remove_nodes_from(nodes_in_match)
    if macro_id_counter > 1:
        print(f"  -> Compressed {macro_id_counter - 1} <CHAMFER_EDGE> macros.")
    return brep_graph

def compress_fillet_macros(brep_graph):
    fillet_template = nx.DiGraph()
    fillet_template.add_node("bsurf1", entity_type="<B_SPLINE_SURFACE_WITH_KNOTS>")
    fillet_template.add_node("bsurf2", entity_type="<B_SPLINE_SURFACE_WITH_KNOTS>")
    # The shared boundary edge (often a B-Spline curve)
    fillet_template.add_node("edge", entity_type="<B_SPLINE_CURVE_WITH_KNOTS>")

    # Topology: Both surfaces point to the exact same shared edge
    fillet_template.add_edges_from([("bsurf1", "edge"), ("bsurf2", "edge")])

    matcher = isomorphism.DiGraphMatcher(brep_graph, fillet_template, node_match=node_match)

    macro_id_counter = 1
    for match in list(matcher.subgraph_isomorphisms_iter()):
        nodes_in_match = list(match.keys())
        # Check if another macro already consumed these nodes
        if not all(n in brep_graph for n in nodes_in_match):
            continue

        inv_match = {v: k for k, v in match.items()}
        fillet_math = np.copy(brep_graph.nodes[inv_match["edge"]]['raw_math'])
        # Assign occ_shape=None so the get_sorting_key function doesn't crash
        # when trying to calculate the surface area of a macro node.
        new_macro_id = f"MACRO_FILLET_{macro_id_counter}"
        brep_graph.add_node(new_macro_id, entity_type="<FILLET_CHAIN>", occ_shape=None, macro_math=fillet_math)
        macro_id_counter += 1

        # Recalculate surrounding topology to the new macro node
        for old_node in nodes_in_match:
            for predecessor in list(brep_graph.predecessors(old_node)):
                if predecessor not in nodes_in_match:
                    brep_graph.add_edge(predecessor, new_macro_id)
            for successor in list(brep_graph.successors(old_node)):
                if successor not in nodes_in_match:
                    brep_graph.add_edge(new_macro_id, successor)

        brep_graph.remove_nodes_from(nodes_in_match)

    if macro_id_counter > 1:
        print(f"  -> Compressed {macro_id_counter - 1} <FILLET_CHAIN> macros.")
    return brep_graph

def compress_cylinder_macros(brep_graph):
    cylinder_template = nx.DiGraph()
    cylinder_template.add_node("cyl", entity_type="<CYLINDRICAL_SURFACE>")
    cylinder_template.add_node("c1", entity_type="<CIRCLE>")
    cylinder_template.add_node("c2", entity_type="<CIRCLE>")

    cylinder_template.add_edge("cyl", "c1")
    cylinder_template.add_edge("cyl", "c2")

    matcher = isomorphism.DiGraphMatcher(brep_graph, cylinder_template, node_match=node_match)
    subgraphs_to_collapse = list(matcher.subgraph_isomorphisms_iter())
    macro_id_counter = 1

    for match in subgraphs_to_collapse:
        nodes_in_match = list(match.keys())
        # Check if another macro already consumed these nodes
        if not all(n in brep_graph for n in nodes_in_match):
            continue

        inv_match = {v: k for k, v in match.items()}
        cyl_math = np.copy(brep_graph.nodes[inv_match["cyl"]]['raw_math'])

        # Assign occ_shape=None so the get_sorting_key function doesn't crash
        # when trying to calculate the surface area of a macro node.
        new_macro_id = f"MACRO_CYL_{macro_id_counter}"
        brep_graph.add_node(new_macro_id, entity_type="<CYLINDER_PRIMITIVE>", occ_shape=None, macro_math=cyl_math)
        macro_id_counter += 1

        # Recalculate surrounding topology to the new macro node
        for old_node in nodes_in_match:
            for predecessor in list(brep_graph.predecessors(old_node)):
                if predecessor not in nodes_in_match:
                    brep_graph.add_edge(predecessor, new_macro_id)
            for successor in list(brep_graph.successors(old_node)):
                if successor not in nodes_in_match:
                    brep_graph.add_edge(new_macro_id, successor)

        brep_graph.remove_nodes_from(nodes_in_match)

    if macro_id_counter > 1:
        print(f"  -> Compressed {macro_id_counter - 1} <CYLINDER_PRIMITIVE> macros.")

    return brep_graph

def compress_sphere_macros(brep_graph):
    sphere_template = nx.DiGraph()
    sphere_template.add_node("sph", entity_type="<SPHERICAL_SURFACE>")
    sphere_template.add_node("c1", entity_type="<CIRCLE>")
    sphere_template.add_edge("sph", "c1")

    matcher = isomorphism.DiGraphMatcher(brep_graph, sphere_template,
                                         node_match=node_match)

    macro_id_counter = 1
    for match in list(matcher.subgraph_isomorphisms_iter()):
        nodes_in_match = list(match.keys())
        # Check if another macro already consumed these nodes
        if not all(n in brep_graph for n in nodes_in_match):
            continue

        inv_match = {v: k for k, v in match.items()}
        sphere_math = np.copy(brep_graph.nodes[inv_match["sph"]]['raw_math'])

        # Assign occ_shape=None so the get_sorting_key function doesn't crash
        # when trying to calculate the surface area of a macro node.
        new_macro_id = f"MACRO_SPH_{macro_id_counter}"
        brep_graph.add_node(new_macro_id, entity_type="<SPHERE_PRIMITIVE>", occ_shape=None, macro_math=sphere_math)
        macro_id_counter += 1

        # Recalculate surrounding topology to the new macro node
        for old_node in nodes_in_match:
            for p in list(brep_graph.predecessors(old_node)):
                if p not in nodes_in_match: brep_graph.add_edge(p, new_macro_id)
            for s in list(brep_graph.successors(old_node)):
                if s not in nodes_in_match: brep_graph.add_edge(new_macro_id, s)

        brep_graph.remove_nodes_from(nodes_in_match)

    if macro_id_counter > 1:
        print(f"  -> Compressed {macro_id_counter - 1} <SPHERE_PRIMITIVE> macros.")

    return brep_graph

def compress_planar_macros(brep_graph):
    pad_template = nx.DiGraph()
    pad_template.add_node("plane", entity_type="<PLANE>")
    pad_template.add_node("loop", entity_type="<EDGE_LOOP>")
    pad_template.add_edge("plane", "loop")

    matcher = isomorphism.DiGraphMatcher(brep_graph, pad_template,
                                         node_match=lambda n1, n2: n1.get('entity_type') == n2.get('entity_type'))

    macro_id_counter = 1
    for match in list(matcher.subgraph_isomorphisms_iter()):
        nodes_in_match = list(match.keys())
        # Check if another macro already consumed these nodes
        if not all(n in brep_graph for n in nodes_in_match): continue

        # Assign occ_shape=None so the get_sorting_key function doesn't crash
        # when trying to calculate the surface area of a macro node.
        new_macro_id = f"MACRO_PAD_{macro_id_counter}"
        inv_match = {v: k for k, v in match.items()}
        brep_graph.add_node(new_macro_id, entity_type="<PLANAR_PAD>", occ_shape=None, pad_math=np.copy(brep_graph.nodes[inv_match["plane"]]['raw_math']))
        macro_id_counter += 1

        # Recalculate surrounding topology to the new macro node
        for old_node in nodes_in_match:
            for p in list(brep_graph.predecessors(old_node)):
                if p not in nodes_in_match: brep_graph.add_edge(p, new_macro_id)
            for s in list(brep_graph.successors(old_node)):
                if s not in nodes_in_match: brep_graph.add_edge(new_macro_id, s)

        brep_graph.remove_nodes_from(nodes_in_match)

    if macro_id_counter > 1:
        print(f"  -> Compressed {macro_id_counter - 1} <PLANAR_PAD> macros.")
    return brep_graph

def compress_macros(compressed_graph):
    # Compress based on complexity / isolation of the macro token

    # 1. Complex tokens
    compressed_graph = compress_hole_macros(compressed_graph)
    # print(f"HOLE-Compressed Graph built with {len(compressed_graph.nodes)} nodes and {len(compressed_graph.edges)} edges.")
    compressed_graph = compress_chamfer_macros(compressed_graph)
    # print(f"CHAMFER-Compressed Graph built with {len(compressed_graph.nodes)} nodes and {len(compressed_graph.edges)} edges.")
    compressed_graph = compress_fillet_macros(compressed_graph)
    # print(f"FILLET-Compressed Graph built with {len(compressed_graph.nodes)} nodes and {len(compressed_graph.edges)} edges.")

    # 2. Isolated tokens
    compressed_graph = compress_cylinder_macros(compressed_graph)
    # print(f"CYLINDER-Compressed Graph built with {len(compressed_graph.nodes)} nodes and {len(compressed_graph.edges)} edges.")
    compressed_graph = compress_sphere_macros(compressed_graph)
    # print(f"SPHERE-Compressed Graph built with {len(compressed_graph.nodes)} nodes and {len(compressed_graph.edges)} edges.")

    # 3. Base features
    compressed_graph = compress_planar_macros(compressed_graph)
    # print(f"PLANAR-Compressed Graph built with {len(compressed_graph.nodes)} nodes and {len(compressed_graph.edges)} edges.")

    return compressed_graph

In [ ]:
def extract_dual_stream(graph):
    visited = set()
    stream_a_tokens = ["<GEO_START>"]
    stream_b_vectors = [np.zeros(128, dtype=np.float32).tolist()]

    root_nodes = [n for n, d in graph.in_degree() if d == 0]
    root_nodes.sort(key=lambda n: get_sorting_key(graph, n))

    def canonical_dfs(node):
        if node in visited: return
        visited.add(node)

        data = graph.nodes[node]
        stream_a_tokens.append(data['entity_type'])

        # Dual Stream logic: Prefer synthesized macro math, fallback to raw math
        vector = data.get('macro_math', data.get('raw_math', np.zeros(128, dtype=np.float32)))
        stream_b_vectors.append(vector.tolist()) # Convert to list for JSON serialization

        children = list(graph.successors(node))
        children.sort(key=lambda c: get_sorting_key(graph, c))
        for child in children: canonical_dfs(child)

    for root in root_nodes: canonical_dfs(root)

    stream_a_tokens.append("<GEO_END>")
    stream_b_vectors.append(np.zeros(128, dtype=np.float32).tolist())

    return stream_a_tokens, stream_b_vectors

## Extract Geometric Vectors (Stream B)

In [ ]:
def parse_step_to_graph(filepath):
    """Parses a STEP file into a base NetworkX DiGraph."""
    reader = STEPControl_Reader()
    status = reader.ReadFile(str(filepath))

    if status != 1:
        print(f"Error reading {filepath}")
        return None

    reader.TransferRoots()

    shape = canonical_normalize_shape(reader.OneShape())

    brep_graph = nx.DiGraph()
    node_counter = 1
    edge_map = {} # Tracks shared edges to prevent duplication
    vertex_map = {} # Tracks shared vertices

    # Traverse all Faces
    face_explorer = TopExp_Explorer(shape, TopAbs_FACE)
    while face_explorer.More():
        face = topods.Face(face_explorer.Current())

        # 1. Map Surface Geometry
        surf_adaptor = BRepAdaptor_Surface(face)
        surf_token = get_surface_token(surf_adaptor.GetType())

        face_node = f"N_{node_counter}"

        brep_graph.add_node(face_node, entity_type=surf_token, occ_shape=face, raw_math=extract_stream_b_vector(face))
        node_counter += 1

        # Traverse Edges bounding this Face
        edge_explorer = TopExp_Explorer(face, TopAbs_EDGE)
        while edge_explorer.More():
            edge = topods.Edge(edge_explorer.Current())
            edge_hash = hash(edge)

            if edge_hash not in edge_map:
                # 2. Map Curve Geometry
                curve_adaptor = BRepAdaptor_Curve(edge)
                curve_token = get_curve_token(curve_adaptor.GetType())

                edge_node = f"N_{node_counter}"
                brep_graph.add_node(edge_node, entity_type=curve_token, occ_shape=edge)
                edge_map[edge_hash] = edge_node
                node_counter += 1
            else:
                edge_node = edge_map[edge_hash]

            # Connect Face to Edge
            brep_graph.add_edge(face_node, edge_node)

            # Traverse Vertices bounding this Edge
            vertex_explorer = TopExp_Explorer(edge, TopAbs_VERTEX)
            while vertex_explorer.More():
                vertex = topods.Vertex(vertex_explorer.Current())
                vertex_hash = hash(vertex)

                if vertex_hash not in vertex_map:
                    # 3. Map Vertex Geometry
                    vert_node = f"N_{node_counter}"
                    brep_graph.add_node(vert_node, entity_type="<VERTEX_POINT>", occ_shape=vertex)
                    vertex_map[vertex_hash] = vert_node
                    node_counter += 1
                else:
                    vert_node = vertex_map[vertex_hash]

                # Connect Edge to Vertex
                brep_graph.add_edge(edge_node, vert_node)
                vertex_explorer.Next()

            edge_explorer.Next()
        face_explorer.Next()

    return brep_graph

In [ ]:
output_file = f"{ROOT_DIR}/cad_hierarchical_dataset.jsonl"
import json

In [ ]:
annotated_files = []
with open(output_file, "r") as f:
    for i, line in enumerate(f):
        line = line.strip()
        if line:
            try:
                obj = json.loads(line)
                file_path = obj["file_path"]
                graph = parse_step_to_graph(file_path)
                print(f"Base Graph built with {len(graph.nodes)} nodes and {len(graph.edges)} edges.")

                graph = precompute_geometric_vectors(graph)
                # Execute it on the graph parsed in the previous step
                compressed_graph = compress_macros(graph)
                print(f"Compressed Graph built with {len(compressed_graph.nodes)} nodes and {len(compressed_graph.edges)} edges.")
                token_stream, vector_stream = extract_dual_stream(compressed_graph)

                obj["token_stream"] = token_stream
                obj["vector_stream"] = vector_stream
                annotated_files.append(obj)
            except json.JSONDecodeError as e:
                print(f"Error decoding JSON on line {i}: {e}")
                continue

In [ ]:
annotated_dataset_filepaths = []
l = len(annotated_files)
if annotated_files:
    for i in range(0, ((l // 100) + 1)):
        filename = DATASET_DIR / f"annotated/annotated_dataset_{(i) * 100}-{(i +1) * 100 - 1 if (i + 1) * 100  < l else l}.jsonl"
        with open(filename, "w") as f:
            f.write("\n".join([json.dumps(dataset_entry, separators=(',', ':')) for dataset_entry in annotated_files[i * 100:(i + 1) * 100]]))
        annotated_dataset_filepaths.append(filename)

In [ ]:
annotated_dataset_filepaths = ["annotated_dataset.jsonl"]

In [ ]:
import torch
import torch.nn as nn
from torch.amp import autocast, GradScaler
from torch.utils.data import Dataset, DataLoader, random_split
from peft import LoraConfig, get_peft_model, TaskType, PeftModel
import json
import matplotlib.pyplot as plt
from tqdm import tqdm
import evaluate
import math
import traceback
import gc

gc.collect()
torch.cuda.empty_cache()

LEARNING_RATE = 5e-5
REPETITION_PENALTY = 1.2

# Assuming the model download finished successfully
MODEL_LOCAL_DIR = MODEL_DIR / "qwen3.5-2B"

# Pass the LOCAL folder path, not the model name
tokenizer = AutoTokenizer.from_pretrained(MODEL_LOCAL_DIR)

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",           # High-accuracy 4-bit type
    bnb_4bit_use_double_quant=True,      # Quantize the quantizers for extra space
    bnb_4bit_compute_dtype=torch.float16 # Math still happens in 16-bit
)

base_model = AutoModelForCausalLM.from_pretrained(MODEL_LOCAL_DIR,
                                                #   quantization_config=bnb_config,
                                                  device_map="auto")
base_model.config.use_cache = False
base_model.gradient_checkpointing_enable()
print(f"Base model: {base_model}")

geometric_tokens = [
    # --- Control & Formatting Tokens ---
    # "<GEO_START>", "<GEO_END>", "<DESCRIBE>", "<PAD>",
    "<GEO_START>", "<GEO_END>", "<DESCRIBE>",

    # --- ISO 10303 Topology Tokens (The Graph) ---
    "<VERTEX_POINT>",
    "<EDGE_CURVE>", "<ORIENTED_EDGE>",
    "<EDGE_LOOP>",
    "<FACE_BOUND>", "<FACE_OUTER_BOUND>", "<ADVANCED_FACE>",
    "<CLOSED_SHELL>", "<OPEN_SHELL>",
    "<MANIFOLD_SOLID_BREP>",

    # --- ISO 10303 Surface Geometry Tokens (The 2D Math) ---
    "<PLANE>",
    "<CYLINDRICAL_SURFACE>", "<CONICAL_SURFACE>",
    "<SPHERICAL_SURFACE>", "<TOROIDAL_SURFACE>",
    "<SURFACE_OF_LINEAR_EXTRUSION>", "<SURFACE_OF_REVOLUTION>",
    "<B_SPLINE_SURFACE_WITH_KNOTS>", "<RATIONAL_B_SPLINE_SURFACE>",

    # --- ISO 10303 Curve Geometry Tokens (The 1D Math) ---
    "<LINE>", "<CIRCLE>", "<ELLIPSE>",
    "<PARABOLA>", "<HYPERBOLA>",
    "<B_SPLINE_CURVE_WITH_KNOTS>", "<RATIONAL_B_SPLINE_CURVE>",

    # --- ISO 10303 Primitive Math Tokens ---
    "<CARTESIAN_POINT>", "<DIRECTION>", "<VECTOR>", "<AXIS2_PLACEMENT_3D>",

    # --- Macro compressed tokens ---
    "<CYLINDER_PRIMITIVE>", # A CYLINDRICAL_SURFACE node connected to exactly two CIRCLE edge nodes
    "<FILLET_CHAIN>", # Two or more B_SPLINE_SURFACE patches that share a common boundary edge and exhibit C^1 (tangential) continuity
    "<THROUGH_HOLE>", # A CYLINDRICAL_SURFACE where its two bounding CIRCLE edges are each connected to separate, distinct PLANE surfaces with opposing normal vectors.
    "<CHAMFER_EDGE>", #A CONICAL_SURFACE or narrow PLANE that bridges two orthogonal PLANE surfaces, bounded by parallel LINE or CIRCLE edges.
    "<SPHERE_PRIMITIVE>", # A SPHERICAL_SURFACE bounded by a single CIRCLE edge (forming a hemisphere or ball joint) or a single vertex pole.
    "<PLANAR_PAD>", # A flat PLANE bounded by an EDGE_LOOP, where every edge in the loop connects perpendicularly to a surrounding "skirt" of PLANE or CYLINDRICAL_SURFACE nodes.

    # Marker for different annotation levels.
    "[L1]", "[L2]", "[L3]",

    # Marker for output.
    "<ANNOTATE>",
]

num_added_toks = tokenizer.add_special_tokens({'additional_special_tokens': geometric_tokens})

# Ensure there is a padding token
if tokenizer.pad_token is None:
    tokenizer.add_special_tokens({'pad_token': '<PAD>'})
    num_added_toks += 1

print(f"Added {num_added_toks} new tokens. New vocab size: {len(tokenizer)} tokens")

# Resize the base model's embedding matrix to accommodate the new tokens
base_model.resize_token_embeddings(len(tokenizer))

# ---------------------------------------------------------
# 1. Dataset Definition
# ---------------------------------------------------------
class CADGeometricDataset(Dataset):
    def __init__(self, paths, tokenizer, max_length=1024):
        self.data = []
        for path in paths:
            with open(path, "r") as f:
                for line in f:
                    line = line.strip()
                    if line:
                        self.data.append(json.loads(line))

        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]

        # Concatenate the semantic geometry tokens with the Ground Truth annotation
        # The model learns to predict the hierarchical target based on the geometry
        full_text = "".join(item["token_stream"]) + "<ANNOTATE>" + item["annotation"] + "<|endoftext|>"

        encoding = self.tokenizer(
            full_text,
            truncation=True,
            max_length=self.max_length,
            padding="max_length",
            return_tensors="pt"
        )

        # Load Stream B (vectors)
        vectors = torch.tensor(item["vector_stream"], dtype=torch.float32)
        padded_vectors = torch.zeros((self.max_length, 128))

        seq_len = min(len(vectors), self.max_length)
        padded_vectors[:seq_len, :] = vectors[:seq_len, :]

        input_ids = encoding["input_ids"].squeeze()
        labels = input_ids.clone()

        # Mask the padding tokens
        labels[input_ids == self.tokenizer.pad_token_id] = -100

        # Mask the Prompt (Everything up to and including <ANNOTATE>)
        annotate_id = self.tokenizer.convert_tokens_to_ids("<ANNOTATE>")
        matches = (input_ids == annotate_id).nonzero(as_tuple=True)[0]

        if len(matches) > 0:
            anno_idx = matches[0].item()
            labels[:anno_idx + 1] = -100 # -100 masks the padding token so the model doesn't train on them.
        else:
            print(f"WARNING: File {item['file_path']} is too big.")
            # Handle large files by preventing training
            labels[:] = -100

            # Keep one single valid token at the end so the CrossEntropyLoss
            # doesn't try to divide by zero and return a NaN loss.
            labels[-1] = self.tokenizer.eos_token_id


        return {
            "input_ids": input_ids,
            "attention_mask": encoding["attention_mask"].squeeze(),
            "geometric_vectors": padded_vectors,
            "labels": labels # Causal LM labeling
        }

# ---------------------------------------------------------
# 2. Dual-Stream Model Architecture
# ---------------------------------------------------------
class GeometricSemanticModel(nn.Module):
    def __init__(self, base_model):
        super().__init__()
        self.base_model = base_model
        # W_p: Projects 128-dim CAD math into 768-dim GPT-2 embedding space
        self.W_p = nn.Linear(128, 768)

    def forward(self, input_ids, geometric_vectors, attention_mask, labels=None):
        # 1. Get standard discrete token embeddings
        # inputs_embeds = self.base_model.transformer.wte(input_ids)
        inputs_embeds = self.base_model.transformer.get_input_embeddings()(input_ids)

        # 2. Project Stream B and fuse with Stream A
        geometric_vectors = geometric_vectors.to(torch.float32)

        # Ensure W_p operates in FP32 to prevent overflow from raw CAD coordinates
        with torch.autocast("cuda", enabled=False):
            geometric_embeds = self.W_p(geometric_vectors)

        # 3. Cast back to match GPT-2's embedding type (FP16 or BF16)
        geometric_embeds = geometric_embeds.to(inputs_embeds.dtype)

        fused_embeds = inputs_embeds + geometric_embeds

        # 3. Pass through the Transformer
        outputs = self.base_model(
            inputs_embeds=fused_embeds,
            attention_mask=attention_mask,
            labels=labels
        )
        return outputs

def annotate(model, tokenizer, start_tokens, prompt_vectors, max_new_tokens=50, device="cuda"):
    model.eval()
    generated_ids = start_tokens.clone()

    # Start with the geometric vectors that match the prompt
    current_vectors = prompt_vectors.clone()

    with torch.no_grad():
        for _ in range(max_new_tokens):
            if generated_ids.shape[1] >= 1024:
                break

            attn_mask = torch.ones_like(generated_ids).to(device)

            outputs = model(
                input_ids=generated_ids,
                geometric_vectors=current_vectors,
                attention_mask=attn_mask
            )

            # Get the logits for the last predicted token
            next_token_logits = outputs.logits[:, -1, :]

            repetition_penalty = REPETITION_PENALTY
            for prev_id in set(generated_ids[0].tolist()):
                if next_token_logits[0, prev_id] < 0:
                    next_token_logits[0, prev_id] *= repetition_penalty
                else:
                    next_token_logits[0, prev_id] /= repetition_penalty

            next_token_id = torch.argmax(next_token_logits, dim=-1).unsqueeze(-1)

            # 1. Grow the Text Stream
            generated_ids = torch.cat([generated_ids, next_token_id], dim=-1)

            zero_vec = torch.zeros((1, 1, 128)).to(device)
            current_vectors = torch.cat([current_vectors, zero_vec], dim=1)

            # Stop early if the model predicts <|endoftext|>
            if next_token_id.item() == tokenizer.eos_token_id:
                break

    return generated_ids

history = {
    "train_loss": [], "val_loss": [],
    "train_bertscore_L1": [], "train_rougeL_L2": [], "train_bleu_L3": [],
    "val_bertscore_L1": [], "val_rougeL_L2": [], "val_bleu_L3": []
}

def parse_hierarchical_levels(text):
    """Slices the generated text into L1, L2, and L3 segments."""
    l1 = re.search(r'\[L1\](.*?)(\[L2\]|$)', text, re.DOTALL)
    l2 = re.search(r'\[L2\](.*?)(\[L3\]|$)', text, re.DOTALL)
    l3 = re.search(r'\[L3\](.*?)$', text, re.DOTALL)

    return {
        "L1": l1.group(1).strip() if l1 else "EMPTY_PREDICTION",
        "L2": l2.group(1).strip() if l2 else "EMPTY_PREDICTION",
        "L3": l3.group(1).strip() if l3 else "EMPTY_PREDICTION"
    }

def train():
    global history
    global base_model
    global tokenizer
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Initializing training on {device}...")

    if isinstance(base_model, PeftModel) or hasattr(base_model, "peft_config"):
        print("Previous PEFT configuration detected. Unloading old adapters...")
        base_model = base_model.unload()

    lora_config = LoraConfig(
        task_type=TaskType.CAUSAL_LM,
        fan_in_fan_out=True,
        r=16,
        lora_alpha=32,
        lora_dropout=0.1,
        target_modules=["c_attn", "c_proj", "c_fc"]
    )
    base_model = get_peft_model(base_model, lora_config)
    model = GeometricSemanticModel(base_model).to(device)

    dataset = CADGeometricDataset(annotated_dataset_filepaths, tokenizer)

    metric_bertscore = evaluate.load("bertscore")
    metric_rouge = evaluate.load("rouge")
    metric_bleu = evaluate.load("bleu")

    train_size = int(0.9 * len(dataset))
    val_size = len(dataset) - train_size
    train_dataset, val_dataset = random_split(dataset, [train_size, val_size])

    train_dataloader = DataLoader(train_dataset, batch_size=1, shuffle=True)
    val_dataloader = DataLoader(val_dataset, batch_size=1, shuffle=False)

    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE)
    epochs = 100
    annotation_token_id = tokenizer.convert_tokens_to_ids("<ANNOTATE>")
    accumulation_steps = 4
    scaler = GradScaler("cuda")

    def evaluate_subset_metrics(dataloader, limit=20):
        """Runs autoregressive generation on a subset and slices metrics hierarchically."""
        model.eval()
        preds_L1, preds_L2, preds_L3 = [], [], []
        refs_L1, refs_L2, refs_L3 = [], [], []

        with torch.no_grad():
            for i, batch in enumerate(dataloader):
                if i >= limit: break

                input_ids = batch["input_ids"].to(device)
                geom_vectors = batch["geometric_vectors"].to(device)

                anno_idx = (input_ids[0] == annotation_token_id).nonzero(as_tuple=True)[0]
                if len(anno_idx) == 0: continue

                prompt_len = anno_idx[0].item() + 1
                prompt_geom_vectors = geom_vectors[:, :prompt_len, :]
                prompt_ids = input_ids[:, :prompt_len]

                generated_ids = annotate(
                    model, tokenizer, prompt_ids, prompt_geom_vectors, max_new_tokens=150, device=device
                )

                # Extract and parse
                new_tokens = generated_ids[0][prompt_len:]
                pred_text = tokenizer.decode(new_tokens, skip_special_tokens=True)

                ref_tokens = input_ids[0][prompt_len:]
                ref_tokens = ref_tokens[ref_tokens != tokenizer.pad_token_id]
                ref_text = tokenizer.decode(ref_tokens, skip_special_tokens=True)

                parsed_preds = parse_hierarchical_levels(pred_text)
                parsed_refs = parse_hierarchical_levels(ref_text)

                preds_L1.append(parsed_preds["L1"])
                preds_L2.append(parsed_preds["L2"])
                preds_L3.append(parsed_preds["L3"])

                refs_L1.append(parsed_refs["L1"])
                refs_L2.append(parsed_refs["L2"])
                refs_L3.append(parsed_refs["L3"])

        # Calculate metrics
        results_bert = metric_bertscore.compute(predictions=preds_L1, references=refs_L1, lang="en", model_type="roberta-base", device="cpu")
        avg_bert = sum(results_bert['f1']) / len(results_bert['f1'])

        results_rouge = metric_rouge.compute(predictions=preds_L2, references=refs_L2)
        avg_rouge = results_rouge['rougeL']

        results_bleu = metric_bleu.compute(predictions=preds_L3, references=[[r] for r in refs_L3])
        avg_bleu = results_bleu['bleu']

        return avg_bert, avg_rouge, avg_bleu

    for epoch in range(epochs):
        model.train()
        total_loss = 0
        progress_bar = tqdm(train_dataloader, desc=f"Epoch {epoch+1}/{epochs} [Train]")
        optimizer.zero_grad()

        for i, batch in enumerate(progress_bar):
            with autocast("cuda", dtype=torch.float16):
                outputs = model(
                    input_ids=batch["input_ids"].to(device),
                    geometric_vectors=batch["geometric_vectors"].to(device),
                    attention_mask=batch["attention_mask"].to(device),
                    labels=batch["labels"].to(device)
                )
                loss = outputs.loss / accumulation_steps
                if math.isnan(loss.item()):
                    print(f"outputs.loss: {outputs.loss}")

            scaled_loss = scaler.scale(loss)
            scaled_loss.backward()

            if (i + 1) % accumulation_steps == 0 or (i + 1) == len(train_dataloader):
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad()

            total_loss += (loss.item() * accumulation_steps)
            progress_bar.set_postfix({"loss": f"{(loss.item() * accumulation_steps):.4f}"})

        # Track Train Loss
        avg_train_loss = total_loss / len(train_dataloader)
        history["train_loss"].append(avg_train_loss)

        # Track Validation Loss
        total_val_loss = 0
        model.eval()
        with torch.no_grad():
            for batch in val_dataloader:
                with autocast("cuda", dtype=torch.float16):
                    outputs = model(
                        input_ids=batch["input_ids"].to(device),
                        geometric_vectors=batch["geometric_vectors"].to(device),
                        attention_mask=batch["attention_mask"].to(device),
                        labels=batch["labels"].to(device)
                    )
                    total_val_loss += outputs.loss.item()

        avg_val_loss = total_val_loss / len(val_dataloader)
        history["val_loss"].append(avg_val_loss)

        print("\nCalculating Hierarchical Metrics...")

        # 1. Train Metrics Subset
        t_bert, t_rouge, t_bleu = evaluate_subset_metrics(train_dataloader, limit=20)
        history["train_bertscore_L1"].append(t_bert)
        history["train_rougeL_L2"].append(t_rouge)
        history["train_bleu_L3"].append(t_bleu)

        # 2. Validation Metrics Subset
        v_bert, v_rouge, v_bleu = evaluate_subset_metrics(val_dataloader, limit=20)
        history["val_bertscore_L1"].append(v_bert)
        history["val_rougeL_L2"].append(v_rouge)
        history["val_bleu_L3"].append(v_bleu)

        print("-" * 65)
        print(f"Epoch {epoch+1} Results: | Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f}")
        print(f"TRAIN METRICS -> L1 (BERT): {t_bert:.4f} | L2 (ROUGE-L): {t_rouge:.4f} | L3 (BLEU-4): {t_bleu:.4f}")
        print(f"VAL METRICS   -> L1 (BERT): {v_bert:.4f} | L2 (ROUGE-L): {v_rouge:.4f} | L3 (BLEU-4): {v_bleu:.4f}")
        print("-" * 65)

    import matplotlib.pyplot as plt
    epochs_range = range(1, epochs + 1)

    fig, axs = plt.subplots(2, 2, figsize=(14, 10))
    fig.suptitle("Dual-Stream Model Training Metrics", fontsize=16)

    axs[0, 0].plot(epochs_range, history["train_loss"], 'b-o', label='Train Loss')
    axs[0, 0].plot(epochs_range, history["val_loss"], 'r-o', label='Val Loss')
    axs[0, 0].set_title("Training vs Validation Loss")
    axs[0, 0].legend()
    axs[0, 0].grid(True, linestyle="--", alpha=0.6)

    axs[0, 1].plot(epochs_range, history["train_bertscore_L1"], 'b--', label='Train L1 (BERT)')
    axs[0, 1].plot(epochs_range, history["val_bertscore_L1"], 'g-o', label='Val L1 (BERT)')
    axs[0, 1].set_title("L1 Functional Classification (BERTScore F1)")
    axs[0, 1].legend()
    axs[0, 1].grid(True, linestyle="--", alpha=0.6)

    axs[1, 0].plot(epochs_range, history["train_rougeL_L2"], 'b--', label='Train L2 (ROUGE-L)')
    axs[1, 0].plot(epochs_range, history["val_rougeL_L2"], 'r-o', label='Val L2 (ROUGE-L)')
    axs[1, 0].set_title("L2 Topological Structure (ROUGE-L)")
    axs[1, 0].legend()
    axs[1, 0].grid(True, linestyle="--", alpha=0.6)

    axs[1, 1].plot(epochs_range, history["train_bleu_L3"], 'b--', label='Train L3 (BLEU)')
    axs[1, 1].plot(epochs_range, history["val_bleu_L3"], 'm-o', label='Val L3 (BLEU)')
    axs[1, 1].set_title("L3 Parametric Precision (BLEU-4)")
    axs[1, 1].legend()
    axs[1, 1].grid(True, linestyle="--", alpha=0.6)

    plt.tight_layout()
    metric_graph_file = "comprehensive_training_metrics.png"
    plt.savefig(metric_graph_file, dpi=300)
    print(f"Metrics plotted and saved to {metric_graph_file}")

    import json
    with open("training_history_metrics.json", "w") as f:
        json.dump(history, f, indent=4)

try:
    train()
except Exception as e:
    with open("training_log.txt", "a+") as f:
        f.write(f"ERROR: {str(e)}\n")
        f.write(traceback.format_exc() + "\n")

# Benchmark Inference